# 02 — Process (Fase 3): Limpieza

**Objetivo:** dejar el dataset limpio y listo para analizar. Cada cambio se documenta en el changelog.

**Reglas:**
- Se trabaja sobre la copia `df_clean`; `data/raw/` no se toca.
- Salida final → `data/clean/hotel_bookings_clean.csv`.
- Cada transformación es una fila de `docs/changelog.md` (qué · por qué · filas antes→después).

Problemas a resolver (detectados en la Fase 2): duplicados · "Undefined" · nulos · outliers · filas sin huéspedes · tipo de fecha.

## 0. Setup

Carga el CSV crudo y crea la **copia de trabajo** `df_clean`. Todo lo demás opera sobre `df_clean`, nunca sobre el original.

In [191]:
# Lee el CSV a un DataFrame con pandas (ruta relativa al notebook)
import pandas as pd

df = pd.read_csv("../data/raw/hotel_bookings.csv")

# Crea una copia del DataFrame original para trabajar en la limpieza de datos
df_clean = df.copy()

## 1. Duplicados exactos (31.994 filas)

**Problema:** 31.994 filas idénticas en las 32 columnas. Sin ID único no se puede confirmar si son error o reservas reales iguales.

**Decisión: se eliminan.** Que dos reservas sean 100 % idénticas (hotel, fechas, lead_time, adr, canal… todo) de forma independiente es muy improbable → casi seguro duplicados de exportación. Mantenerlos sesgaría las tasas de cancelación.

**Caveat:** sin ID único no hay certeza absoluta.

In [192]:
# Elimina filas duplicadas, manteniendo la primera aparición

df_clean.drop_duplicates(keep='first', inplace=True)

# Comprueba filas duplicadas después de eliminar duplicados
print(len(df))              # cuántas filas hay
print(len(df_clean))        # cuántas filas quedan después de eliminar duplicados
print(df_clean.duplicated().sum())        # cuántas filas duplicadas quedan después de eliminar duplicados

119390
87396
0


## 2. "Undefined" → NaN (nulos disfrazados)

**Problema:** `meal` (1.169), `distribution_channel` (5) y `market_segment` (2) usan el texto "Undefined" en vez de NaN, así que `isnull()` no los detecta.

**Decisión: se convierte "Undefined" a NaN** en esas columnas para unificar los faltantes antes de tratarlos.

**Matiz en `meal`:** `SC` y `Undefined` significan ambos "sin paquete de comida"; una alternativa válida sería fusionar `meal` Undefined → `SC` en lugar de NaN.

In [193]:
# Reemplazar valores undefined por NaN

cols_undef = ["meal", "distribution_channel", "market_segment"]
df_clean[cols_undef] = df_clean[cols_undef].replace("Undefined", pd.NA)
df_clean[cols_undef].isna().sum()

for col in ['meal', 'distribution_channel', 'market_segment', 'country']:
    print(f"=== {col} ===")
    print(df_clean[col].value_counts(dropna=False))
    print()

=== meal ===
meal
BB     67978
SC      9481
HB      9085
NaN      492
FB       360
Name: count, dtype: int64

=== distribution_channel ===
distribution_channel
TA/TO        69141
Direct       12988
Corporate     5081
GDS            181
NaN              5
Name: count, dtype: int64

=== market_segment ===
market_segment
Online TA        51618
Offline TA/TO    13889
Direct           11804
Groups            4942
Corporate         4212
Complementary      702
Aviation           227
NaN                  2
Name: count, dtype: int64

=== country ===
country
PRT    27453
GBR    10433
FRA     8837
ESP     7252
DEU     5387
       ...  
NCL        1
KIR        1
SDN        1
ATF        1
SLE        1
Name: count, Length: 178, dtype: int64



## 3. Nulos reales (tras el paso 2)

Decisión por columna. Ninguna es protagonista de las preguntas guía, así que se prioriza simplicidad y honestidad:

| Columna | Nulos | Decisión recomendada |
|---|---|---|
| `company` | 112.593 (94 %) | **Eliminar la columna** — 94 % vacía, inservible. |
| `agent` | 16.340 (14 %) | NaN = “sin agencia” (reserva directa) → se rellena con `0` (es un ID, no se imputa con la media). |
| `country` | 488 (0,4 %) | Poco impacto → se rellena con “Unknown”. |
| `children` | 4 | Se rellena con `0` (lo más probable: sin niños). |

In [194]:
#  Eliminamos columna "company" por tener demasiados valores faltantes (94% de los datos)
df_clean.drop(columns=['company'], inplace=True)

# Sustituimos los valores faltantes de "agent" por 0, ya que es un identificador numérico.
df_clean['agent'] = df_clean['agent'].fillna(0)

# Sustituimos los valores faltantes de "country" por "Unknown", ya que es una variable categórica.
df_clean['country'] = df_clean['country'].fillna("Unknown")

# Sustituimos los valores faltantes de "children" por 0, ya que es un número entero que representa la cantidad de niños, y es razonable asumir que los valores faltantes corresponden a 0 niños.

df_clean['children'] = df_clean['children'].fillna(0)



## 4. Outliers / valores imposibles

| Caso | Decisión recomendada |
|---|---|
| `adr` = −6.38 (1 fila) | **Eliminar la fila** — tarifa negativa = error claro. |
| `adr` = 5.400 (1 fila) | Extremo (p75 = 126), valor aislado → se elimina como probable error. |
| `adults` 55 · `children` 10 · `babies` 10 · `stays_in_week_nights` 50 | Extremos pero no imposibles (grupos, estancias largas). Se conservan salvo los aislados con salto brusco. |

Regla: se elimina solo lo **imposible** (adr negativo) y los outliers aislados con salto brusco; lo raro-pero-posible se mantiene (documentado).

In [195]:
# Eliminamos la fila con adr negativo, ya que no tiene sentido tener un precio negativo.

df_clean = df_clean[df_clean['adr'] >= 0]

# Comprobamos que no hay valores faltantes ni filas con adr negativo después de la limpieza
df_clean[df_clean['adr'] < 0]

# Comprobamos adr extremos ordenamos en orden descendente por adr y mostramos las 10 filas con adr más alto.
df_clean.sort_values('adr', ascending=False).head(10)

# Comprobamos valores extremos en adults ordenamos en orden descendente por adults y mostramos las 10 filas con más adultos.
df_clean.sort_values('adults', ascending=False).head(10)[['children','adults','babies','adr','hotel','arrival_date_month']]

# Comprobamos valores extremos en children ordenamos en orden descendente por children y mostramos las 10 filas con más children.
df_clean.sort_values('children', ascending=False).head(10)[['children','adults','babies','adr','hotel','arrival_date_month']]

# Comprobamos valores extremos en babies ordenamos en orden descendente por babies y mostramos las 10 filas con más babies.
df_clean.sort_values('babies', ascending=False).head(10)[['children','adults','babies','adr','hotel','arrival_date_month']]

# Comprobamos valores extremos en stays_in_week_nights ordenamos en orden descendente por stays_in_week_nights y mostramos las 10 filas con más stays_in_week_nights.
df_clean.sort_values('stays_in_week_nights', ascending=False).head(10)[['stays_in_week_nights','children','adults','babies','adr','hotel','arrival_date_month']]

outliers = (
    (df_clean['children'] >= 10) |
    (df_clean['babies'] >= 9)    |
    (df_clean['adr'] >= 5400)   
)
df_clean = df_clean[~outliers]



## 5. Filas sin huéspedes (180)

**Problema:** 180 filas con `adults + children + babies = 0`. Una reserva sin ocupantes no sirve para analizar demanda.

**Decisión: se eliminan** (~0,2 % del dataset). Registro inválido. También se eliminan las reservas con 0 adultos y algún menor (un menor no puede ser el único huésped).

In [196]:
# Comprobar reservas con 0 adultos ordenamos en orden ascendente por children y mostramos las 10 filas con 0 adultos.
df_clean[df_clean['adults'] == 0].sort_values('children', ascending=True).head(10)[['children','adults','babies','adr','hotel','arrival_date_month']]

# Eliminamos las filas con 0 adultos, 0 niños y 0 bebés, ya que no tiene sentido tener una reserva sin ningún huésped.
df_clean = df_clean[(df_clean['adults'] + df_clean['children'] + df_clean['babies']) > 0]

# Eliminamos reservas con 0 adultos pero con niños/bebés (menor como único huésped = inválido)
df_clean = df_clean[~((df_clean['adults'] == 0) & ((df_clean['children'] + df_clean['babies']) > 0))]

## 6. `reservation_status_date` → fecha

**Problema:** está como texto, no como fecha.

**Decisión: se convierte a datetime** (permite operaciones temporales).

⚠️ **Leakage:** `reservation_status` y `reservation_status_date` revelan el resultado (Canceled/No-Show = cancelación) y solo se conocen DESPUÉS de la reserva. Se limpian, pero **no se usan como predictores** en la Fase 4.

In [197]:
# Convertimos la columna "reservation_status_date" a tipo datetime
df_clean['reservation_status_date'] = pd.to_datetime(df_clean['reservation_status_date'], errors='coerce')

## 7. Verificación post-limpieza

Comprobaciones de que la limpieza quedó bien:
- Duplicados = 0
- Nulos: tratados o documentados
- Tipos correctos (`reservation_status_date` es fecha)
- Rangos sanos (ej. `adr` ≥ 0)
- Filas finales vs originales (119.390 → 86.999)

In [ ]:
# Comprobamos duplicados después de la limpieza
print(len(df_clean))        # cuántas filas quedan después de la limpieza
print(df_clean.duplicated().sum())        # cuántas filas duplicadas quedan después de la limpieza

# Eliminamos filas duplicadas, manteniendo la primera aparición
df_clean = df_clean.drop_duplicates()
print(df_clean.duplicated().sum())   # debe dar 0
print(len(df_clean))                 # ~86999

87006
7
0
86999


In [199]:

# Comprobamos nulos después de la limpieza
print(df_clean.isna().sum())


hotel                               0
is_canceled                         0
lead_time                           0
arrival_date_year                   0
arrival_date_month                  0
arrival_date_week_number            0
arrival_date_day_of_month           0
stays_in_weekend_nights             0
stays_in_week_nights                0
adults                              0
children                            0
babies                              0
meal                              492
country                             0
market_segment                      2
distribution_channel                5
is_repeated_guest                   0
previous_cancellations              0
previous_bookings_not_canceled      0
reserved_room_type                  0
assigned_room_type                  0
booking_changes                     0
deposit_type                        0
agent                               0
days_in_waiting_list                0
customer_type                       0
adr         

In [200]:
# Comprobamos el tipo de datos de reservation_status_date
print(df_clean['reservation_status_date'].dtype)

datetime64[us]


In [201]:
# Comprobamos rangos sanos
df_clean.describe().T.round(2).loc[['children','babies','adr']]

,count,mean,min,25%,50%,75%,max,std
children,86999.0,0.134036,0.0,0.0,0.0,0.0,3.0,0.445416
babies,86999.0,0.010621,0.0,0.0,0.0,0.0,2.0,0.104178
adr,86999.0,106.524836,0.0,72.25,98.33,134.1,510.0,51.898554


In [202]:
# Comprobamos reservas con 0 adultos ordenamos en orden ascendente por children y mostramos las 10 filas con 0 adultos.
df_clean[df_clean['adults'] == 0].sort_values('children', ascending=True).head(10)[['children','adults','babies','adr','hotel','arrival_date_month']]


,children,adults,babies,adr,hotel,arrival_date_month


## 8. Guardar dataset limpio

Exportamos `df_clean` a `../data/clean/hotel_bookings_clean.csv` (sin el índice). Este CSV es el que se analiza en la Fase 4 (notebook 03, con SQL/DuckDB).

In [203]:
# Exportamos el DataFrame limpio a un nuevo CSV `../data/clean/hotel_bookings_clean.csv`
df_clean.to_csv("../data/clean/hotel_bookings_clean.csv", index=False)

## 9. Resumen de cambios

**Resultado: 119.390 → 86.999 filas (−32.391).** Changelog completo en `docs/changelog.md`.

| Transformación (orden de ejecución) | Resultado |
|---|---|
| Duplicados exactos eliminados | 119.390 → 87.396 |
| `"Undefined"` → NaN (meal, distribution_channel, market_segment) | nulos unificados |
| `company` eliminada · `agent`→0 · `country`→"Unknown" · `children`→0 | nulos tratados |
| Outliers eliminados: adr −6.38, adr 5.400, children 10, babies 9–10 | extremos aislados |
| Sin ocupantes (total=0) + 0 adultos con menores (219) eliminadas | ocupación inválida |
| `reservation_status_date` → datetime | tipo corregido |
| Re-dedup final (−7): 87.006 → **86.999** | final |
| Mantenidos con criterio: adults/stays (gradiente) · NaN meal/market/channel | decisión consciente |

Dataset limpio guardado → `data/clean/hotel_bookings_clean.csv`.